In [171]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [172]:
import torch
import gc
from datasets import load_dataset
from evaluate import load
from transformers import AutoModelForCausalLM, AutoTokenizer
from tabulate import tabulate
from tqdm import tqdm
import re
import textwrap

model_label_map = {
    "Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder": "LLaMa",
    "lmsys/vicuna-13b-v1.5": "Vicuna"
}

model_names = list(model_label_map.keys())

all_generations_table = []
problem_candidate_list = {model_name: {} for model_name in model_names}

for model_name in model_names:
  code_eval = load("code_eval")

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  use_chat_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None

  print(f"{model_name} uses {use_chat_template}")

  if "vicuna" in model_name.lower():
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            load_in_4bit=True,
            torch_dtype=torch.float16
        )
  else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )

  model.eval()

  for i in tqdm([6, 9], desc=f"{model_label_map[model_name]}"):
      print(f"Inside loop with problem {i}")

      dataset = load_dataset("openai_humaneval")['test'].select([i])

      for problem in dataset:
          entry_point = problem['entry_point']
          test_code = problem['test']
          problem_prompt = problem['prompt']

          sig_match = re.search(r'def.*:', problem_prompt, flags=re.MULTILINE)
          function_signature = sig_match.group(0).strip() if sig_match else None

          if i == 6:
            prompt = f"""
            Generate exactly one assert statement. Do not include any code, function signatures, or explanations.
            Design an assert statement to test the following function:
            Function Signature: {function_signature}
            Function Description: A function that outputs a list of the deepest level of nesting of parenthesis from the input 'paren_string'.
            MUST generate a working assert statement
            Boolean comparison assert result with some modification happening to paren_string DO NOT USE calling parse_nested_parens

            Return a modified version of:
            assert result == paren_string
            """
          elif i == 9:
            prompt = f"""
            Generate exactly one assert statement. Do not include any code, function signatures, or explanations.
            Design an assert statement to test the following function:
            Function Signature: {function_signature}
            Function Description: A function that outputs a list of the rolling maximum from the input 'number'.
            MUST generate a working assert statement
            Boolean comparison assert result with some modification happening to numbers DO NOT USE rolling_max

            Return a modified version of:
            assert result == numbers
            """
          problem_candidates = []
          for _ in range(5):
            messages = [
                {"role": "system", "content": "Follow the user's instructions exactly. Output exactly ONE assert statement, nothing else. Cannot use rolling_max or parse_nested_parens in assertion. Can only use result."},
                {"role": "user", "content": prompt}
            ]

            if use_chat_template:
                chat_text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    template=None
                )
            else:
              chat_text = (
                "USER: Follow the instructions exactly. Output one assert statement only, nothing else. Cannot use rolling_max or parse_nested_parens in assertion. Can only use result.\n"
                f"USER: {prompt}\n"
                "ASSISTANT:"
              )
            inputs = tokenizer(
                chat_text,
                return_tensors="pt"
            ).to(device)

            input_len = inputs["input_ids"].shape[1]

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=30,
                    temperature=0.5,
                    top_p=0.95,
                    do_sample=True,
                    eos_token_id=tokenizer.eos_token_id
                )

            generated_ids = outputs[0][input_len:]
            generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
            generated_code = re.sub(r'^\s*(```python|assistant|system|user|###)[:\s]*\n*', '', generated_text, flags=re.IGNORECASE)
            problem_candidates.append(generated_code)

          all_generations_table.append({
              "Problem ID": i + 1,
              "LLM": model_label_map[model_name],
              "Gen 1": problem_candidates[0],
              "Gen 2": problem_candidates[1],
              "Gen 3": problem_candidates[2],
              "Gen 4": problem_candidates[3],
              "Gen 5": problem_candidates[4],
          })
          problem_candidate_list[model_name][i] = problem_candidates

del model, tokenizer
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder uses True


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LLaMa:   0%|          | 0/2 [00:00<?, ?it/s]

Inside loop with problem 6


LLaMa:  50%|█████     | 1/2 [00:08<00:08,  8.92s/it]

...

Vicuna:   0%|          | 0/2 [00:00<?, ?it/s]

Inside loop with problem 6


Vicuna:  50%|█████     | 1/2 [00:11<00:11, 11.28s/it]

Inside loop with problem 9


Vicuna: 100%|██████████| 2/2 [00:25<00:00, 12.95s/it]


In [173]:
for model_name, problems in problem_candidate_list.items():
    for problem_id, candidates in problems.items():
        for gen_idx, candidate in enumerate(candidates, 1):
            print(f"{model_name}, {problem_id}, {gen_idx}:{candidate}\n\n")

Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 6, 1:assert result == [1, 2, 2, 1, 2, 1]


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 6, 2:```python
assert result == [0, 1, 2, 3, 0, 1, 2


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 6, 3:assert parse_nested_parens("((())())()()()") == [3, 3, 1, 1


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 6, 4:assert result == [1, 1, 2, 3, 1, 2, 3, 4


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 6, 5:assert parse_nested_parens("((())())()()") == [2, 2, 1, 1]


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 9, 1:```python
assert result == [1, 2, 3, 3, 4, 4, 4


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 9, 2:assert result == [9, 9, 9, 9, 9, 9, 9, 9


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 9, 3:```python
assert result == [1, 2, 3, 4, 5, 6, 7


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 9, 4:assert result == [1, 2, 3, 4, 5, 6, 7, 8


Solshine/Meta-Llama-3.1-8B-Instruct-Python-Coder, 9, 5:assert res

In [174]:
from tabulate import tabulate

table_str = tabulate(all_generations_table, headers="keys", tablefmt="grid")
print(table_str)

with open("generated_assertions_for_formal_specifications.csv", "w", encoding="utf-8") as f:
    f.write(table_str)

from google.colab import files
files.download("/content/generated_assertions_for_formal_specifications.csv")

+--------------+--------+------------------------------------------------------------------+------------------------------------------------------------------+------------------------------------------------------------------+------------------------------------------+------------------------------------------------------------------+
|   Problem ID | LLM    | Gen 1                                                            | Gen 2                                                            | Gen 3                                                            | Gen 4                                    | Gen 5                                                            |
+==============+========+==================================================================+==================================================================+==================================================================+==========================================+=========================================================

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>